# Infraestructuras Computacionales para el Procesamiento de datos masivos

**Ejercicio 1, Módulo 2 Tema 2 — Consumidor de streaming**

Este cuaderno lee los ficheros JSON que genera `M2T3_productor.ipynb` desde `input/crypto_prices` y calcula métricas con Spark Structured Streaming.

- generación de eventos: `M2T3_productor.ipynb`
- lectura y métricas: este cuaderno


## 1. Preparación del entorno

Ejecuta esta celda para crear las carpetas necesarias. Si ya existen, y tienen contenido es mejor eliminarlas

In [ ]:
import os
import shutil

INPUT_DIR = os.path.abspath("input/crypto_prices")
OUTPUT_DIR = os.path.abspath("output/crypto_prices_parquet")
CHECKPOINT_DIR = os.path.abspath("checkpoint/crypto_prices")

for path in (INPUT_DIR, OUTPUT_DIR, CHECKPOINT_DIR):
    shutil.rmtree(path, ignore_errors=True)
    os.makedirs(path, exist_ok=True)

print("Directorio de trabajo:", os.getcwd())
print("Carpetas preparadas y limpiadas:")
print("-", INPUT_DIR)
print("-", OUTPUT_DIR)
print("-", CHECKPOINT_DIR)


## 2. Inicialización de SparkSession

En Spark Structured Streaming se recomienda trabajar con `SparkSession`, ya que es el punto de entrada moderno para usar DataFrames, SQL y streaming estructurado.

Si estás ejecutando el ejercicio en modo local, `local[*]` utiliza todos los núcleos disponibles de la máquina.

In [ ]:
!pip install pyspark==3.5.1

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder     .appName("CoinGeckoFileStreaming")     .master("local[*]")     .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

spark

## 3. Definición del esquema

Cuando Spark lee JSON en streaming, es buena práctica definir explícitamente el esquema.

Esto evita que Spark tenga que inferirlo continuamente y hace el proceso más estable.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

schema = StructType([
    StructField("event_time", TimestampType(), True),
    StructField("coin", StringType(), True),
    StructField("currency", StringType(), True),
    StructField("price", DoubleType(), True)
])

schema

## 4. Lectura del flujo de ficheros JSON

Spark leerá como streaming todos los nuevos ficheros que aparezcan en la carpeta `input/crypto_prices`.

Importante: los ficheros los genera `M2T3_productor.ipynb`; este cuaderno solo los consume y calcula métricas.


In [ ]:
stream_df = spark.readStream     .schema(schema)     .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss'Z'")     .json(INPUT_DIR)

stream_df.printSchema()


## 5. Consulta streaming básica en consola

Esta primera consulta muestra los eventos recibidos por Spark en la consola.

Sirve para comprobar que Spark detecta correctamente los nuevos ficheros JSON.

In [ ]:
raw_query = stream_df.writeStream     .format("console")     .option("checkpointLocation", os.path.join(CHECKPOINT_DIR, "raw_console"))     .outputMode("append")     .option("truncate", "false")     .trigger(availableNow=True)     .start()
raw_query.awaitTermination()


## 6. Agregaciones por ventana temporal

Ahora se calculan métricas por criptomoneda en ventanas de 1 minuto:

- precio medio,
- precio máximo,
- precio mínimo,
- número de eventos procesados.

La marca de agua (`watermark`) permite gestionar eventos que lleguen tarde.

In [ ]:
from pyspark.sql.functions import col, window, avg, max, min, count

metrics_df = stream_df     .withWatermark("event_time", "1 minute")     .groupBy(
        window(col("event_time"), "1 minute"),
        col("coin")
    )     .agg(
        avg("price").alias("avg_price"),
        max("price").alias("max_price"),
        min("price").alias("min_price"),
        count("*").alias("num_events")
    )

metrics_df.printSchema()

## 7. Visualización de métricas en consola

La salida en modo `complete` muestra la tabla agregada completa para que no aparezca vacía.


In [ ]:
metrics_query = metrics_df.writeStream     .option("checkpointLocation", os.path.join(CHECKPOINT_DIR, "metrics_console"))     .outputMode("complete")     .format("console")     .option("truncate", "false")     .trigger(availableNow=True)     .start()
metrics_query.awaitTermination()


## 8. Escritura de eventos en Parquet

Parquet es un formato columnar eficiente, muy usado en arquitecturas de datos.

En streaming, Spark necesita una carpeta de checkpoint para mantener el estado del proceso y poder recuperarse ante fallos.

In [ ]:
parquet_query = stream_df.writeStream     .format("parquet")     .option("path", OUTPUT_DIR)     .option("checkpointLocation", os.path.join(CHECKPOINT_DIR, "parquet"))     .outputMode("append")     .trigger(availableNow=True)     .start()
parquet_query.awaitTermination()


In [ ]:
# Para detener la consulta:
parquet_query.stop()

## 9. Lectura posterior de los datos Parquet

Cuando la consulta de escritura haya procesado algunos eventos, puedes detenerla y leer los datos almacenados en Parquet como un DataFrame estático.

In [ ]:
# Ejecuta esta celda después de detener parquet_query
# parquet_query.stop()

historical_df = spark.read.parquet(OUTPUT_DIR)
historical_df.show(truncate=False)
historical_df.groupBy("coin").count().show()

## 10. Detener consultas activas

Es importante detener las consultas streaming cuando se termina el ejercicio para liberar recursos y la instancia de spark.

In [ ]:
for query in spark.streams.active:
    print("Deteniendo consulta:", query.name, query.id)
    query.stop()

print("Consultas activas:", spark.streams.active)

spark.stop()


## 11. Ejercicios

1. Añadir más criptomonedas a la lista `COINS`.
2. Cambiar la moneda de referencia de `eur` a `usd`.
3. Calcular ventanas de 5 minutos en lugar de 1 minuto.
4. Crear una alerta cuando el precio de Bitcoin supere un umbral definido.
5. Guardar las métricas agregadas en Parquet en lugar de guardar los eventos originales.
6. Añadir una columna `source` con valor `coingecko`.
7. Comparar el número de eventos esperados con el número de eventos realmente procesados.